# 03 — Feature Engineering

We turn each movie's overview, genres, keywords, top cast, and director into a single `tags` string —
a bag-of-words representation that `CountVectorizer` and cosine similarity (Notebook 4) will operate on.

**Input:** `data/processed_movies.csv`
**Output:** `data/final_movies.csv`


In [1]:
import pandas as pd
import ast
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

df = pd.read_csv("../data/processed_movies.csv")
for col in ["genres_list", "keywords_list", "cast_list", "production_companies_list"]:
    df[col] = df[col].apply(ast.literal_eval)

df.shape


(4799, 15)

## 1. Build the `tags` column

We combine:
- `overview` split into individual words
- `genres_list`
- `keywords_list`
- `cast_list` (top 3 billed actors)
- `director`

Multi-word names (e.g. "Sam Worthington") are collapsed into single tokens ("samworthington") so the
vectorizer treats a full name as one feature instead of splitting it into two generic words.

In [2]:
def collapse(name: str) -> str:
    return name.replace(" ", "").lower()


df["overview_tokens"] = df["overview"].apply(lambda x: str(x).split())
df["genres_tokens"] = df["genres_list"].apply(lambda lst: [collapse(g) for g in lst])
df["keywords_tokens"] = df["keywords_list"].apply(lambda lst: [collapse(k) for k in lst])
df["cast_tokens"] = df["cast_list"].apply(lambda lst: [collapse(c) for c in lst])
df["director_tokens"] = df["director"].apply(lambda d: [collapse(d)] if isinstance(d, str) and d else [])

df["tags_raw"] = (
    df["overview_tokens"] + df["genres_tokens"] + df["keywords_tokens"]
    + df["cast_tokens"] + df["director_tokens"]
)
df["tags_raw"] = df["tags_raw"].apply(lambda lst: " ".join(lst))

print("BEFORE (raw tags), example:\n")
print(df.loc[0, "tags_raw"][:300], "...")


BEFORE (raw tags), example:

In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance spac ...


## 2. NLP cleaning pipeline

Lowercase -> tokenize -> remove stopwords -> strip special characters -> stem with `PorterStemmer`.
This is the same logic reused by `app/nlp_pipeline.py` in the Streamlit app, kept consistent so a movie's
offline-computed tags match anything typed live in the search box.

In [3]:
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)

STOPWORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()


def clean_and_stem(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)


df["tags"] = df["tags_raw"].apply(clean_and_stem)


## 3. Before / after examples

In [4]:
for i in [0, 5, 25]:
    print(f"=== {df.loc[i, 'title']} ===")
    print("BEFORE:", df.loc[i, "tags_raw"][:200], "...")
    print("AFTER :", df.loc[i, "tags"][:200], "...")
    print()


=== Avatar ===
BEFORE: In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy ...
AFTER : 22nd centuri parapleg marin dispatch moon pandora uniqu mission becom torn follow order protect alien civil action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetra ...

=== Spider-Man 3 ===
BEFORE: The seemingly invincible Spider-Man goes up against an all-new crop of villain – including the shape-shifting Sandman. While Spider-Man’s superpowers are altered by an alien organism, his alter ego, P ...
AFTER : seemingli invinc spider man goe new crop villain includ shape shift sandman spider man superpow alter alien organ alter ego peter parker deal nemesi eddi brock also get caught love triangl fantasi act ...

=== Titanic ===
BEFORE: 84 years later, a 101-year-old woman named Rose DeWitt Bukater tells the story to her 

## 4. Assemble the final feature table

We keep the cleaned `tags` plus the display metadata the Streamlit app needs (genres, cast, director,
rating, year) so the app can render a result card without re-parsing anything.

In [5]:
final_cols = [
    "movie_id", "title", "tags", "overview", "genres_list", "keywords_list",
    "cast_list", "director", "release_year", "vote_average", "vote_count", "popularity", "runtime",
]
final_df = df[final_cols].copy()
final_df.rename(columns={
    "genres_list": "genres", "keywords_list": "keywords", "cast_list": "cast",
}, inplace=True)

final_df.reset_index(drop=True, inplace=True)
final_df.head(3)


,movie_id,title,tags,overview,genres,keywords,cast,director,release_year,vote_average,vote_count,popularity,runtime
0,19995,Avatar,22nd centuri parapleg marin dispatch moon pand...,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron,2009,7.2,11800,150.437577,162.0
1,285,Pirates of the Caribbean: At World's End,captain barbossa long believ dead come back li...,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski,2007,6.9,4500,139.082615,169.0
2,206647,Spectre,cryptic messag bond past send trail uncov sini...,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes,2015,6.3,4466,107.376788,148.0


## 5. Save

In [6]:
final_df.to_csv("../data/final_movies.csv", index=False)
print(f"Saved data/final_movies.csv with {len(final_df)} rows.")
print(f"Average tag length: {final_df['tags'].str.split().apply(len).mean():.1f} tokens/movie.")


Saved data/final_movies.csv with 4799 rows.


Average tag length: 44.6 tokens/movie.
